# Magazine Article OCR Pipeline — notebook_article_ocr

Extracts, chunks, and indexes genealogy magazine articles from Google Drive using Vertex AI Gemini.
Designed to be run interactively or as a Databricks Job whenever new article images are added.

**Source table:** `workspace.staging_google_drive.articles`
**Volume path:** `/Volumes/genealogy/staging_google_drive/articles/`
**Output tables:**
- `workspace.genealogy.silver_research_article` — one row per article
- `workspace.genealogy.silver_article_chunk` — one row per chunk (CDF enabled — vector index source)
- `workspace.genealogy.silver_article_ocr_log` — one row per article attempt (idempotency)

**Filename convention:**
- Lead file (multi-page): `{magazine}_{issue}_p{start}-{end}.jpg` e.g. `wdytya_219_p15-19.jpg`
- Member files: `{magazine}_{issue}_p{page}.jpg` e.g. `wdytya_219_p16.jpg`
- Single-page: `{magazine}_{issue}_p{page}.jpg` e.g. `wdytya_220_p44.jpg`

Magazine codes: `wdytya` → Who Do You Think You Are?, `ft` → Family Tree

**Change log:**
- v1.0: Initial implementation. Single multi-part Gemini call per article group.
        CDF-enabled silver_article_chunk feeds vector index.


In [0]:
%pip install "google-genai==1.64.0" "pydantic<2.12" tenacity databricks-vectorsearch


In [0]:
import json
import re
import time
import traceback
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path

from tenacity import retry, wait_random_exponential, stop_after_attempt, retry_if_exception_type
import google.api_core.exceptions
import base64

from google import genai
from google.genai import types
from google.oauth2 import service_account

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, TimestampType
)


In [0]:
# ── GCP / Gemini ──────────────────────────────────────────────────────────
GCP_PROJECT   = "genealogy-488213"
GCP_LOCATION  = "global"              # Latest Gemini models require global endpoint
VERTEX_MODEL  = "gemini-3.1-pro-preview"
SECRET_SCOPE  = "genealogy"
SECRET_KEY    = "gcp_service_account_json"

# Seconds between article groups — keeps RPM well under Vertex AI quota.
# Each article group is one Gemini call regardless of page count.
INTER_REQUEST_DELAY = 4

# ── Unity Catalog ──────────────────────────────────────────────────────────
CATALOG        = "workspace"
STAGING_SCHEMA = "staging_google_drive"
OUTPUT_SCHEMA  = "genealogy"

SOURCE_TABLE   = f"{CATALOG}.{STAGING_SCHEMA}.articles"
ARTICLE_TBL    = f"{CATALOG}.{OUTPUT_SCHEMA}.silver_research_article"
CHUNK_TBL      = f"{CATALOG}.{OUTPUT_SCHEMA}.silver_article_chunk"
LOG_TBL        = f"{CATALOG}.{OUTPUT_SCHEMA}.silver_article_ocr_log"

# Volume path where Fivetran syncs article images
VOLUME_PATH    = "/Volumes/workspace/staging_google_drive/articles"

# ── Vector Search ──────────────────────────────────────────────────────────
# One endpoint per Free Edition account. Index name follows catalog.schema.name pattern.
VS_ENDPOINT_NAME = "genealogy_vs_endpoint"
VS_INDEX_NAME    = "workspace.genealogy.silver_article_chunk_index"
# Embedding column in silver_article_chunk that will be indexed.
# We use Databricks-managed embeddings (gte-large-en-v1.5 via Foundation Model APIs).
VS_SOURCE_COL    = "chunk_text"
VS_PRIMARY_KEY   = "chunk_id"

# ── Magazine code → full name ──────────────────────────────────────────────
MAGAZINE_NAMES = {
    "wdytya": "Who Do You Think You Are?",
    "ft":     "Family Tree",
}

# ── Topic tag vocabulary (closed list — passed verbatim to Gemini prompt) ──
TOPIC_TAGS = [
    "Baptisms", "Deaths", "Marriages", "Census", "Military", "DNA",
    "Newspapers", "Brick Walls", "Scottish", "Irish", "Welsh",
    "Pre-1837", "Occupations", "Migration", "Places", "General",
]

print(f"Source table : {SOURCE_TABLE}")
print(f"Article table: {ARTICLE_TBL}")
print(f"Chunk table  : {CHUNK_TBL}")
print(f"Log table    : {LOG_TBL}")
print(f"VS index     : {VS_INDEX_NAME}")


In [0]:
# Secret is stored as base64-encoded JSON (encoded before saving to Databricks secret)
sa_json_b64 = dbutils.secrets.get(scope=SECRET_SCOPE, key=SECRET_KEY)
sa_json     = base64.b64decode(sa_json_b64).decode("utf-8")
sa_info     = json.loads(sa_json)

credentials = service_account.Credentials.from_service_account_info(
    sa_info,
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)

# google-genai client replaces deprecated vertexai.generative_models (removed June 2026)
client = genai.Client(
    vertexai=True,
    project=GCP_PROJECT,
    location=GCP_LOCATION,
    credentials=credentials,
)

print(f"google-genai client initialised. Model: {VERTEX_MODEL}")


In [0]:
# CREATE TABLE IF NOT EXISTS is idempotent — safe to rerun.
# ALTER TABLE to enable Change Data Feed on silver_article_chunk so the
# Databricks Vector Search index can sync incrementally from it.

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {ARTICLE_TBL} (
        article_id       STRING    COMMENT 'Unique article ID parsed from lead filename, e.g. wdytya_219_p15',
        title            STRING    COMMENT 'Article title extracted by Gemini',
        magazine_code    STRING    COMMENT 'Magazine code parsed from filename, e.g. wdytya',
        magazine_name    STRING    COMMENT 'Full magazine name mapped from code',
        issue_number     STRING    COMMENT 'Issue number parsed from filename',
        start_page       INT       COMMENT 'First page number parsed from lead filename',
        topic_tags       STRING    COMMENT 'Comma-separated topic tags assigned by Gemini',
        source_files     STRING    COMMENT 'Comma-separated volume paths of all page images',
        processed_at     TIMESTAMP COMMENT 'When this article was written'
    )
    USING DELTA
    COMMENT 'One row per magazine article processed by the article OCR pipeline'
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CHUNK_TBL} (
        chunk_id         STRING    COMMENT 'Unique chunk ID, e.g. wdytya_219_p15_001',
        article_id       STRING    COMMENT 'FK to silver_research_article',
        chunk_index      INT       COMMENT '0-based position within article',
        heading          STRING    COMMENT 'Section heading if present, else NULL',
        chunk_text       STRING    COMMENT 'Combined heading + body text — this is the vector index source',
        topic_tags       STRING    COMMENT 'Denormalised from parent article',
        magazine_code    STRING    COMMENT 'Denormalised',
        magazine_name    STRING    COMMENT 'Denormalised',
        issue_number     STRING    COMMENT 'Denormalised',
        start_page       INT       COMMENT 'Denormalised'
    )
    USING DELTA
    COMMENT 'One row per article chunk — CDF enabled so vector index syncs incrementally'
""")

# Enable Change Data Feed — required by Databricks Vector Search standard endpoint
# so it can track inserts/updates/deletes and keep the index in sync.
try:
    spark.sql(f"ALTER TABLE {CHUNK_TBL} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
    print(f"CDF enabled on {CHUNK_TBL}")
except Exception as e:
    print(f"CDF already set or minor issue (non-fatal): {e}")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {LOG_TBL} (
        lead_file_path   STRING    COMMENT 'Volume path of the lead file — used as idempotency key',
        article_id       STRING    COMMENT 'Derived article ID',
        status           STRING    COMMENT 'success / error / skipped',
        page_count       INT       COMMENT 'Number of page images sent to Gemini',
        model_used       STRING    COMMENT 'Vertex AI model used',
        processed_at     TIMESTAMP COMMENT 'When this attempt completed',
        error_message    STRING    COMMENT 'Populated if status is error'
    )
    USING DELTA
    COMMENT 'Processing log — one row per article attempt, used for idempotency'
""")

print("Tables ready.")


In [0]:
# ── Filename parsing ──────────────────────────────────────────────────────
#
# Lead file:   {magazine}_{issue}_p{start}-{end}.jpg  → multi-page article
# Member file: {magazine}_{issue}_p{page}.jpg          → subsequent pages
# Single page: {magazine}_{issue}_p{page}.jpg          → article_id = wdytya_220_p44
#
# Examples:
#   wdytya_219_p15-19.jpg  → article_id=wdytya_219_p15, pages 15-19 (lead)
#   wdytya_219_p16.jpg     → member of above
#   wdytya_220_p44.jpg     → single-page, article_id=wdytya_220_p44

# Regex for lead file: magazine_issue_pSTART-END.jpg
LEAD_PATTERN   = re.compile(r'^([a-z]+)_([^_]+)_p(\d+)-(\d+)\.', re.IGNORECASE)
# Regex for any page file: magazine_issue_pPAGE.jpg
MEMBER_PATTERN = re.compile(r'^([a-z]+)_([^_]+)_p(\d+)\.', re.IGNORECASE)


def parse_lead_filename(filename: str) -> dict | None:
    """
    Parse a lead filename. Returns metadata dict or None if not a lead file.
    A lead file has a page range: wdytya_219_p15-19.jpg
    """
    m = LEAD_PATTERN.match(Path(filename).name)
    if not m:
        return None
    mag_code, issue, start_page, end_page = m.groups()
    article_id = f"{mag_code.lower()}_{issue}_p{start_page}"
    return {
        "article_id":    article_id,
        "magazine_code": mag_code.lower(),
        "magazine_name": MAGAZINE_NAMES.get(mag_code.lower(), mag_code),
        "issue_number":  issue,
        "start_page":    int(start_page),
        "end_page":      int(end_page),
        "page_numbers":  list(range(int(start_page), int(end_page) + 1)),
    }


def parse_member_filename(filename: str) -> dict | None:
    """
    Parse a single-page or member filename. Returns metadata dict or None.
    e.g. wdytya_219_p16.jpg → magazine=wdytya, issue=219, page=16
    """
    # Must NOT match the lead pattern (which has a dash range)
    if LEAD_PATTERN.match(Path(filename).name):
        return None
    m = MEMBER_PATTERN.match(Path(filename).name)
    if not m:
        return None
    mag_code, issue, page = m.groups()
    return {
        "magazine_code": mag_code.lower(),
        "magazine_name": MAGAZINE_NAMES.get(mag_code.lower(), mag_code),
        "issue_number":  issue,
        "page_number":   int(page),
    }


@dataclass
class ArticleGroup:
    """Represents one article — a lead file plus its member page files in order."""
    article_id:    str
    magazine_code: str
    magazine_name: str
    issue_number:  str
    start_page:    int
    lead_path:     str                     # Volume path of the lead image
    page_paths:    list = field(default_factory=list)  # All page paths in order


def extract_json(text: str) -> str:
    """Extract outermost JSON object from Gemini response, ignoring preamble/fences."""
    match = re.search(r'\{.*\}', text, re.DOTALL)
    return match.group(0) if match else text


In [0]:
# ── Discover all files in the articles staging table ─────────────────────
# Fivetran syncs article images into staging_google_drive.articles.
# We build the volume path from the Fivetran file path column.
# Cross-reference the log table to skip already-processed lead files.

source_df = spark.sql(f"""
    SELECT
        file_id,
        '{VOLUME_PATH}/' || _fivetran_file_path AS file_path,
        _fivetran_file_path                      AS relative_path
    FROM {SOURCE_TABLE}
    WHERE _fivetran_deleted = false
""")

processed_leads = spark.sql(f"""
    SELECT lead_file_path
    FROM {LOG_TBL}
    WHERE status = 'success'
""")

all_files = (
    source_df
    .orderBy("relative_path")
    .collect()
)

processed_set = {r.lead_file_path for r in processed_leads.collect()}

print(f"Total files in staging : {len(all_files)}")
print(f"Already processed      : {len(processed_set)}")

# ── Group files into ArticleGroups ─────────────────────────────────────────
# Pass 1: identify all lead files and create groups
groups: dict[str, ArticleGroup] = {}
for row in all_files:
    fname = Path(row.file_path).name
    lead_meta = parse_lead_filename(fname)
    if lead_meta:
        if row.file_path not in processed_set:
            groups[lead_meta["article_id"]] = ArticleGroup(
                article_id    = lead_meta["article_id"],
                magazine_code = lead_meta["magazine_code"],
                magazine_name = lead_meta["magazine_name"],
                issue_number  = lead_meta["issue_number"],
                start_page    = lead_meta["start_page"],
                lead_path     = row.file_path,
                page_paths    = [row.file_path],  # lead page is first
            )

# Pass 2: add member pages to existing groups (sorted by page number)
# Also handle single-page articles (member file with no matching lead)
for row in all_files:
    fname = Path(row.file_path).name
    member_meta = parse_member_filename(fname)
    if not member_meta:
        continue
    # Find which group this page belongs to by matching magazine + issue + page range
    for grp in groups.values():
        if (grp.magazine_code == member_meta["magazine_code"]
                and grp.issue_number == member_meta["issue_number"]
                and member_meta["page_number"] > grp.start_page  # lead page already added
                and row.file_path not in grp.page_paths):
            grp.page_paths.append(row.file_path)
            break
    else:
        # Single-page article — no lead file found; treat as its own group
        article_id = f"{member_meta['magazine_code']}_{member_meta['issue_number']}_p{member_meta['page_number']}"
        if article_id not in groups and row.file_path not in processed_set:
            groups[article_id] = ArticleGroup(
                article_id    = article_id,
                magazine_code = member_meta["magazine_code"],
                magazine_name = member_meta["magazine_name"],
                issue_number  = member_meta["issue_number"],
                start_page    = member_meta["page_number"],
                lead_path     = row.file_path,
                page_paths    = [row.file_path],
            )

# Sort page_paths within each group by page number
for grp in groups.values():
    grp.page_paths.sort(key=lambda p: int(re.search(r'_p(\d+)', Path(p).stem).group(1)))

candidate_groups = list(groups.values())
print(f"Article groups to process: {len(candidate_groups)}")
for g in candidate_groups:
    print(f"  {g.article_id}  ({len(g.page_paths)} page(s))  {g.magazine_name}")


In [0]:
# ── Gemini prompt builder ─────────────────────────────────────────────────

def build_article_prompt(magazine_name: str) -> str:
    """
    Build the Gemini extraction prompt for a magazine article.

    Key design principles (mirrors OCR pipeline):
    - JSON-only instruction is both positive and negative to suppress preamble/fences
    - Topic vocabulary list appears immediately before the output skeleton
    - Output skeleton appears at end — LLMs weight end-of-prompt heavily
    - All page images sent as a single call; Gemini reads across pages naturally
    """
    vocab_list = ", ".join(f'"{t}"' for t in TOPIC_TAGS)
    return f"""You are reading a genealogy magazine article from {magazine_name}.
The article may span multiple pages — all page images are provided.

Extract the article title, assign topic tags, and split the article into logical sections (chunks).
Each chunk should represent a coherent sub-topic or paragraph group — aim for 150–400 words per chunk.
Extract the key information and meaning from each section. 
Rewrite in clear prose — paraphrasing is acceptable and preferred.
Do not reproduce copyrighted material verbatim.

## Output Format

Your response must be a single JSON object and nothing else.
Do not include any text, explanation, or commentary before or after the JSON.
Do not wrap the JSON in markdown code fences.
Do not use ```json or ```.
Your entire response must be valid JSON starting with {{ and ending with }}.

Choose topic_tags only from this list: {vocab_list}
Assign 1–4 tags that best describe the article's subject matter.

{{
  "title": "The full article title as printed",
  "magazine_confirmed": "The magazine name as printed on the page",
  "topic_tags": ["tag1", "tag2"],
  "chunks": [
    {{"heading": "Section heading if present, else null", "text": "body text of this section"}},
    {{"heading": null, "text": "introductory prose with no heading"}}
  ]
}}"""


# ── Gemini call with retry ─────────────────────────────────────────────────

def _call_gemini_article_once(page_paths: list, prompt: str) -> tuple:
    """
    Send all article page images in a single Gemini call.
    Each page is a separate types.Part.from_bytes() part — Gemini reads them in order.
    Returns (result_dict, usage_dict).
    """
    parts = []
    for path in page_paths:
        with open(path, "rb") as f:
            image_bytes = f.read()
        # All article images are JPEGs from the Google Drive export
        parts.append(types.Part.from_bytes(data=image_bytes, mime_type="image/jpeg"))
    parts.append(prompt)  # text prompt as final part

    response = client.models.generate_content(
        model=VERTEX_MODEL,
        contents=parts,
        config=types.GenerateContentConfig(
            temperature=1.0, # increase temp from 0.1 to avoid recitation error
            max_output_tokens=65536,
            thinking_config=types.ThinkingConfig(thinking_level="LOW"),
            http_options=types.HttpOptions(timeout=180000),  # 3 min — multi-page articles
        ),
    )
    print(response.text)
    um = response.usage_metadata
    usage = {
        "prompt_tokens":   getattr(um, "prompt_token_count",     0) or 0,
        "output_tokens":   getattr(um, "candidates_token_count", 0) or 0,
        "thinking_tokens": getattr(um, "thoughts_token_count",   0) or 0,
        "total_tokens":    getattr(um, "total_token_count",      0) or 0,
    }

    if not response.candidates:
        print(f"  No candidates — prompt_feedback: {response.prompt_feedback}")
        return {"_safety_blocked": True}, usage

    candidate = response.candidates[0]
    print(f"  finish_reason: {candidate.finish_reason}")
    print(f"  safety_ratings: {getattr(candidate, 'safety_ratings', 'none')}")

    if not candidate.content or not candidate.content.parts:
        return {
            "_safety_blocked": True,
            "_finish_reason": str(getattr(candidate, "finish_reason", "unknown")),
            "_safety_ratings": str(getattr(candidate, "safety_ratings", "unknown")),
        }, usage

    raw_text = response.text.strip()
    raw_text = extract_json(raw_text)
    try:
        return json.loads(raw_text), usage
    except json.JSONDecodeError:
        return {
            "_parse_error": True,
            "_raw": raw_text[:500],
        }, usage


@retry(
    retry=retry_if_exception_type((
        google.api_core.exceptions.ResourceExhausted,
        google.api_core.exceptions.TooManyRequests,
        google.api_core.exceptions.DeadlineExceeded,
        google.api_core.exceptions.Cancelled,
    )),
    wait=wait_random_exponential(multiplier=1, max=60),
    stop=stop_after_attempt(5),
    before_sleep=lambda rs: print(
        f"    Rate limited — attempt {rs.attempt_number} failed, retrying..."
    ),
)
def call_gemini_article(page_paths: list, prompt: str) -> tuple:
    """Gemini call with tenacity exponential backoff on 429/504 errors."""
    return _call_gemini_article_once(page_paths, prompt)


In [0]:
# ── PySpark schemas ──────────────────────────────────────────────────────

ARTICLE_SCHEMA = StructType([
    StructField("article_id",    StringType(),    True),
    StructField("title",         StringType(),    True),
    StructField("magazine_code", StringType(),    True),
    StructField("magazine_name", StringType(),    True),
    StructField("issue_number",  StringType(),    True),
    StructField("start_page",    IntegerType(),   True),
    StructField("topic_tags",    StringType(),    True),
    StructField("source_files",  StringType(),    True),
    StructField("processed_at",  TimestampType(), True),
])

CHUNK_SCHEMA = StructType([
    StructField("chunk_id",      StringType(),    True),
    StructField("article_id",    StringType(),    True),
    StructField("chunk_index",   IntegerType(),   True),
    StructField("heading",       StringType(),    True),
    StructField("chunk_text",    StringType(),    True),
    StructField("topic_tags",    StringType(),    True),
    StructField("magazine_code", StringType(),    True),
    StructField("magazine_name", StringType(),    True),
    StructField("issue_number",  StringType(),    True),
    StructField("start_page",    IntegerType(),   True),
])

LOG_SCHEMA = StructType([
    StructField("lead_file_path", StringType(),    True),
    StructField("article_id",     StringType(),    True),
    StructField("status",         StringType(),    True),
    StructField("page_count",     IntegerType(),   True),
    StructField("model_used",     StringType(),    True),
    StructField("processed_at",   TimestampType(), True),
    StructField("error_message",  StringType(),    True),
])


def write_article_row(row: dict):
    df = spark.createDataFrame([row], schema=ARTICLE_SCHEMA)
    df.write.format("delta").mode("append").saveAsTable(ARTICLE_TBL)


def write_chunk_rows(rows: list):
    if not rows:
        return
    df = spark.createDataFrame(rows, schema=CHUNK_SCHEMA)
    df.write.format("delta").mode("append").saveAsTable(CHUNK_TBL)


def write_log_row(row: dict):
    df = spark.createDataFrame([row], schema=LOG_SCHEMA)
    df.write.format("delta").mode("append").saveAsTable(LOG_TBL)


# ── Processing loop ───────────────────────────────────────────────────────
# Writes to all three tables immediately after each article — safe against
# cancellation. On the next run, already-logged success articles are skipped.

total_groups  = len(candidate_groups)
success_count = 0
error_count   = 0
total_chunks  = 0

for i, grp in enumerate(candidate_groups, 1):
    now = datetime.now(timezone.utc)
    print(f"[{i}/{total_groups}] {grp.article_id}  ({len(grp.page_paths)} page(s))")

    prompt = build_article_prompt(grp.magazine_name)

    try:
        result, usage = call_gemini_article(grp.page_paths, prompt)

        if result.get("_safety_blocked"):
            print(f"  ✗ Blocked — finish_reason={result.get('_finish_reason')}  ratings={result.get('_safety_ratings')}")
            write_log_row({
                "lead_file_path": grp.lead_path,
                "article_id":     grp.article_id,
                "status":         "error",
                "page_count":     len(grp.page_paths),
                "model_used":     VERTEX_MODEL,
                "processed_at":   now,
                "error_message":  "Response blocked by Gemini safety filters",
            })
            error_count += 1
            continue

        if result.get("_parse_error"):
            print(f"  ✗ JSON parse error: {result.get('_raw', '')[:100]}")
            write_log_row({
                "lead_file_path": grp.lead_path,
                "article_id":     grp.article_id,
                "status":         "error",
                "page_count":     len(grp.page_paths),
                "model_used":     VERTEX_MODEL,
                "processed_at":   now,
                "error_message":  f"JSON parse error. Raw: {result.get('_raw', '')[:500]}",
            })
            error_count += 1
            continue

        title      = result.get("title", "Untitled")
        topic_tags = ", ".join(result.get("topic_tags", []))
        chunks_raw = result.get("chunks", [])

        # Write article row
        write_article_row({
            "article_id":    grp.article_id,
            "title":         title,
            "magazine_code": grp.magazine_code,
            "magazine_name": grp.magazine_name,
            "issue_number":  grp.issue_number,
            "start_page":    grp.start_page,
            "topic_tags":    topic_tags,
            "source_files":  ", ".join(grp.page_paths),
            "processed_at":  now,
        })

        # Write chunk rows — heading + text combined into chunk_text
        chunk_rows = []
        for idx, chunk in enumerate(chunks_raw):
            heading    = chunk.get("heading")  # may be None
            body_text  = chunk.get("text", "").strip()
            chunk_text = f"{heading}\n\n{body_text}" if heading else body_text
            chunk_rows.append({
                "chunk_id":      f"{grp.article_id}_{idx+1:03d}",
                "article_id":    grp.article_id,
                "chunk_index":   idx,
                "heading":       heading,
                "chunk_text":    chunk_text,
                "topic_tags":    topic_tags,
                "magazine_code": grp.magazine_code,
                "magazine_name": grp.magazine_name,
                "issue_number":  grp.issue_number,
                "start_page":    grp.start_page,
            })
        write_chunk_rows(chunk_rows)

        # Write success log
        write_log_row({
            "lead_file_path": grp.lead_path,
            "article_id":     grp.article_id,
            "status":         "success",
            "page_count":     len(grp.page_paths),
            "model_used":     VERTEX_MODEL,
            "processed_at":   now,
            "error_message":  None,
        })

        total_chunks += len(chunk_rows)
        print(f"  ✓ Done  title=\"{title[:60]}\"  chunks={len(chunk_rows)}  tags={topic_tags}")
        success_count += 1

    except Exception as e:
        error_msg = f"{type(e).__name__}: {e}\n{traceback.format_exc()}"
        print(f"  ✗ Error: {error_msg[:200]}")
        write_log_row({
            "lead_file_path": grp.lead_path,
            "article_id":     grp.article_id,
            "status":         "error",
            "page_count":     len(grp.page_paths),
            "model_used":     VERTEX_MODEL,
            "processed_at":   now,
            "error_message":  error_msg[:2000],
        })
        error_count += 1

    if i < total_groups:
        time.sleep(INTER_REQUEST_DELAY)

print(f"\n{'='*50}")
print(f"Processing complete: {success_count} succeeded, {error_count} errors, {total_chunks} chunks written")


In [0]:
# ── Vector Search index sync ─────────────────────────────────────────────
#
# First run: the index must already exist (create it once manually in the
# Databricks UI or via the SDK — see instructions printed below if not found).
# Subsequent runs: this cell triggers a sync of new/updated chunks into the index.
#
# Free Edition allows one VS endpoint and one VS unit. The index uses
# Databricks-managed embeddings (no separate embedding endpoint needed).

from databricks.vector_search.client import VectorSearchClient

vs_client = VectorSearchClient(
    workspace_url   = 'https://dbc-23cc95ff-20a0.cloud.databricks.com', # spark.conf.get("spark.databricks.workspaceUrl"),
    service_principal_client_id     = None,  # uses notebook's own auth
    disable_notice  = True,
)

try:
    index = vs_client.get_index(
        endpoint_name = VS_ENDPOINT_NAME,
        index_name    = VS_INDEX_NAME,
    )
    print(f"Index found: {VS_INDEX_NAME}")
    print(f"Index status: {index.describe().get('status', {}).get('detailed_state', 'unknown')}")

    if success_count > 0:
        print("Triggering sync...")
        index.sync()

        # Poll until index is ONLINE_NO_PENDING_UPDATE (up to 10 min)
        import time as _time
        for attempt in range(40):
            status = index.describe().get("status", {}).get("detailed_state", "")
            print(f"  [{attempt+1}/40] {status}")
            if status in ("ONLINE_NO_PENDING_UPDATE", "ONLINE"):
                print(f"  ✓ Index online and up to date.")
                break
            _time.sleep(15)
        else:
            print("  ⚠ Index sync did not complete within 10 min — check Databricks UI")
    else:
        print("No new articles processed — skipping index sync.")

except Exception as e:
    if "does not exist" in str(e).lower() or "not found" in str(e).lower():
        print(f"⚠ Index or endpoint not found. Create them once using these steps:")
        print(f"  1. In Databricks UI → Compute → Vector Search → Create endpoint: '{VS_ENDPOINT_NAME}'")
        print(f"  2. Then run this SDK call once to create the index:")
        print(f"")
        print(f"     vs_client.create_delta_sync_index(")
        print(f"         endpoint_name         = '{VS_ENDPOINT_NAME}',")
        print(f"         index_name            = '{VS_INDEX_NAME}',")
        print(f"         source_table_name     = '{CHUNK_TBL}',")
        print(f"         pipeline_type         = 'TRIGGERED',")
        print(f"         primary_key           = '{VS_PRIMARY_KEY}',")
        print(f"         embedding_source_column = '{VS_SOURCE_COL}',")
        print(f"         embedding_model_endpoint_name = 'databricks-gte-large-en',")
        print(f"     )")
        print(f"  3. Then re-run this notebook to load and index articles.")
    else:
        print(f"⚠ Vector search error (non-fatal — articles are written, index sync failed): {e}")


In [0]:
print(f"\n{'='*50}")
print(f"PIPELINE SUMMARY")
print(f"{'='*50}")
print(f"Articles processed  : {total_groups}")
print(f"Succeeded           : {success_count}")
print(f"Errors              : {error_count}")
print(f"Total chunks written: {total_chunks}")
print()

# Show article table summary
spark.sql(f"""
    SELECT magazine_name, COUNT(*) AS articles, SUM(SIZE(SPLIT(topic_tags, ','))) AS tag_count
    FROM {ARTICLE_TBL}
    GROUP BY magazine_name
    ORDER BY articles DESC
""").show(truncate=False)

# Show chunk table summary
spark.sql(f"""
    SELECT magazine_name, COUNT(*) AS chunks
    FROM {CHUNK_TBL}
    GROUP BY magazine_name
    ORDER BY chunks DESC
""").show(truncate=False)

# Show this run's log entries
if candidate_groups:
    run_article_ids = ", ".join(f"'{g.article_id}'" for g in candidate_groups)
    spark.sql(f"""
        SELECT article_id, status, page_count, processed_at, error_message
        FROM {LOG_TBL}
        WHERE article_id IN ({run_article_ids})
        ORDER BY processed_at
    """).show(truncate=80)
